## ABUK — LSTM Neural Network

The previous experiment used a deep feed-forward neural network to
predict the next-day return of ABUK using the previous 20 daily returns.

In this experiment, we replace the feed-forward network with an LSTM.

The purpose is to test whether explicitly modeling the sequential
relationship between historical returns improves the prediction.

To make the comparison fair, the LSTM uses:

- The same ABUK stock
- The same 20-day lookback window
- The same chronological train/test split
- The same next-day return target
- The same trading rule

The final results will be compared with the previous deep neural network.

In [ ]:
import sys, os

while not os.path.isdir("src") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed
from tradinglab.simulator import PortfolioSimulator

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

feed = DataFeed.from_dir(
    "data/egx",
    symbols=["ABUK"]
)

print("symbols:", feed.symbols)
print("number of assets:", feed.n_assets)
print("number of days:", feed.n_days)

In [ ]:
abuk_returns = feed.returns[:, 0]

print("ABUK returns shape:", abuk_returns.shape)

In [ ]:
feed.returns[:, 0]

In [ ]:
LOOKBACK = 20

Xs = []
ys = []

for i in range(LOOKBACK, len(abuk_returns)):

    Xs.append(
        abuk_returns[i-LOOKBACK:i]
    )

    ys.append(
        abuk_returns[i]
    )

Xs = np.array(Xs, dtype=np.float32)
ys = np.array(ys, dtype=np.float32)

print("X shape:", Xs.shape)
print("y shape:", ys.shape)

In [ ]:
Xs = Xs.reshape(-1, LOOKBACK, 1)

print("LSTM X shape:", Xs.shape)

In [ ]:
split = int(len(Xs) * 0.7)

Xtr_lstm = torch.tensor(Xs[:split])
ytr_lstm = torch.tensor(ys[:split])

Xte_lstm = torch.tensor(Xs[split:])
yte_lstm = torch.tensor(ys[split:])

print("training samples:", len(Xtr_lstm))
print("testing samples:", len(Xte_lstm))

In [ ]:
class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):

        out, (hidden, cell) = self.lstm(x)

        # Take the output from the final timestep
        last_output = out[:, -1, :]

        return self.head(last_output).squeeze(-1)

In [ ]:
torch.manual_seed(0)

lstm_model = LSTMModel()

print(lstm_model)

In [ ]:
optimizer_lstm = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

train_hist_lstm = []
test_hist_lstm = []

EPOCHS = 300

for epoch in range(EPOCHS):

    lstm_model.train()

    optimizer_lstm.zero_grad()

    predictions = lstm_model(Xtr_lstm)

    loss = nn.functional.mse_loss(
        predictions,
        ytr_lstm
    )

    loss.backward()

    optimizer_lstm.step()

    train_hist_lstm.append(loss.item())

    # Test evaluation
    lstm_model.eval()

    with torch.no_grad():

        test_predictions = lstm_model(Xte_lstm)

        test_loss = nn.functional.mse_loss(
            test_predictions,
            yte_lstm
        )

        test_hist_lstm.append(test_loss.item())

In [ ]:
print(
    f"LSTM final train loss: {train_hist_lstm[-1]:.6f}"
)

print(
    f"LSTM final test loss:  {test_hist_lstm[-1]:.6f}"
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    train_hist_lstm,
    label="LSTM train loss"
)

plt.plot(
    test_hist_lstm,
    label="LSTM test loss"
)

plt.yscale("log")

plt.xlabel("Epoch")
plt.ylabel("MSE")

plt.title("ABUK LSTM — Training vs Test Loss")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
lstm_model.eval()

with torch.no_grad():

    lstm_predictions = (
        lstm_model(Xte_lstm)
        .numpy()
    )

actual_returns = yte_lstm.numpy()

print("prediction shape:", lstm_predictions.shape)
print("actual shape:", actual_returns.shape)

In [ ]:
SHOW_N = 150

plt.figure(figsize=(12, 4))

plt.plot(
    actual_returns[:SHOW_N],
    label="actual next-day return"
)

plt.plot(
    lstm_predictions[:SHOW_N],
    label="LSTM prediction",
    linestyle="--"
)

plt.axhline(0, linewidth=0.8)

plt.title("ABUK — LSTM Prediction vs Actual")

plt.xlabel("Test day")
plt.ylabel("Return")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
lstm_weights = np.where(
    lstm_predictions > 0,
    1.0,
    0.0
)

lstm_strategy_returns = (
    lstm_weights * actual_returns
)

In [ ]:
lstm_total_return = (
    np.prod(1 + lstm_strategy_returns) - 1
)

print(
    f"LSTM strategy return: {lstm_total_return:+.2%}"
)

In [ ]:
def max_drawdown(returns):

    curve = np.cumprod(1 + returns)

    peak = np.maximum.accumulate(curve)

    drawdown = (peak - curve) / peak

    return drawdown.max()

In [ ]:
lstm_drawdown = max_drawdown(
    lstm_strategy_returns
)

print(
    f"LSTM max drawdown: {lstm_drawdown:.2%}"
)

In [ ]:
'''SHOW_N = 150

plt.figure(figsize=(12, 5))

plt.plot(
    actual_returns[:SHOW_N],
    label="Actual"
)

plt.plot(
    predictions[:SHOW_N],
    label="Deep NN",
    linestyle="--"
)

plt.plot(
    lstm_predictions[:SHOW_N],
    label="LSTM",
    linestyle=":"
)

plt.axhline(0, linewidth=0.8)

plt.title("ABUK — Deep NN vs LSTM Predictions")

plt.xlabel("Test day")
plt.ylabel("Next-day return")

plt.legend()
plt.grid(alpha=0.3)

plt.show()'''